In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler
df = pd.read_csv("magic04.csv")

In [ ]:
df["class"] = (df["class"] == "g").astype(int)
df.head()

In [ ]:
cols = df.columns.tolist()

for label in cols[:-1]:
    gamma_data  = df[df["class"] == 1][label]
    hadron_data = df[df["class"] == 0][label]

    plt.hist(gamma_data,  color='blue', label='Gamma',  alpha=0.7, density=True)
    plt.hist(hadron_data, color='red',  label='Hadron', alpha=0.7, density=True)

    plt.title(label)
    plt.ylabel('Probability')
    plt.xlabel(label)
    plt.legend()
    plt.show()

Train,validation,test datasets...

In [ ]:
train = df.sample(frac=0.6)
temp  = df.drop(train.index)

valid = temp.sample(frac=0.5)
test  = temp.drop(valid.index)


In [ ]:
def scale_dataset(dataframe, oversample=False):
    X = dataframe[dataframe.columns[:-1]].values
    Y = dataframe[dataframe.columns[-1]].values

    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    if oversample:
        oversampler = RandomOverSampler()
        X, Y = oversampler.fit_resample(X, Y)

    data = np.hstack((X, np.reshape(Y, (-1, 1))))
    return data, X, Y

In [ ]:
train_data, X_train, Y_train = scale_dataset(train, oversample=True)
valid_data, X_valid, Y_valid  = scale_dataset(valid)
test_data,  X_test,  Y_test   = scale_dataset(test)

In [ ]:
train = df.sample(frac=0.6)
temp = df.drop(train.index)

valid = temp.sample(frac=0.)
test = temp.drop(valid.index)

In [ ]:
print(len(train[train["class"] == 1]))
print(len(train[train["class"] == 0]))

KNN (K-nearest neighbour)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=3)
knn_model.fit(X_train,Y_train)

In [ ]:
y_pred = knn_model.predict(X_test)

In [ ]:
print(classification_report(Y_test,y_pred))

Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB

In [ ]:
nb_model = GaussianNB()
nb_model = nb_model.fit(X_train,Y_train)

In [ ]:
y_pred = nb_model.predict(X_test)
print(classification_report(Y_test,y_pred))

Log Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
lg_model = LogisticRegression()
lg_model = lg_model.fit(X_train,Y_train)

In [ ]:
y_pred = lg_model.predict(X_test)
print(classification_report(Y_test,y_pred))

SVM

In [ ]:
from sklearn.svm import SVC

In [ ]:
svm_model = SVC()
svm_model = svm_model.fit(X_train, Y_train)

In [ ]:
y_pred = svm_model.predict(X_test)
print(classification_report(Y_test,y_pred))

Neural Net

In [ ]:
import tensorflow as tf

In [ ]:
import tensorflow as tf

def plot_history(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

    ax1.plot(history.history['loss'], label='loss')
    ax1.plot(history.history['val_loss'], label='val_loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Binary crossentropy')
    ax1.grid(True)
    ax1.legend()

    ax2.plot(history.history['accuracy'], label='accuracy')
    ax2.plot(history.history['val_accuracy'], label='val_accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.grid(True)
    ax2.legend()

    plt.show()

In [ ]:
def train_model(X_train, Y_train, num_nodes, dropout_prob, lr, batch_size, epochs):
    nn_model = tf.keras.Sequential([
        tf.keras.layers.Dense(num_nodes, activation='relu', input_shape=(10,)),
        tf.keras.layers.Dropout(dropout_prob),
        tf.keras.layers.Dense(num_nodes, activation='relu'),
        tf.keras.layers.Dropout(dropout_prob),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

    nn_model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    history = nn_model.fit(
        X_train, Y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=0.2,
        verbose=0
    )

    return nn_model, history

In [ ]:
least_val_loss = float('inf')
least_loss_model = None
epochs = 100

for num_nodes in [16, 32, 64]:
    for dropout_prob in [0, 0.2]:
        for lr in [0.1, 0.005, 0.001]:
            for batch_size in [32, 64, 128]:
                print(f"{num_nodes} nodes, dropout {dropout_prob}, lr {lr}, batch size {batch_size}")
                model, history = train_model(X_train, Y_train, num_nodes, dropout_prob, lr, batch_size, epochs)
                plot_history(history)
                val_loss = model.evaluate(X_valid, Y_valid)[0]

                if val_loss < least_val_loss:
                    least_val_loss = val_loss
                    least_loss_model = model

In [54]:
least_loss_model.predict(X_test)

119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step


array([[1.1144564e-05],
       [9.1649640e-01],
       [9.7491157e-01],
       ...,
       [3.4278193e-01],
       [8.9347690e-02],
       [3.2334615e-08]], shape=(3804, 1), dtype=float32)